# Week 7 Exercise: Getting Better Predictions from Our Fine-Tuned Model

We trained Llama 3.2-3B with QLoRA on ~20K product listings to predict prices.
Model: `mcaleb/price-2026-03-05_04.52.30-lite`

The plan:
1. Start with the basic prediction (greedy decoding)
2. Try two tricks to get better results, without retraining
3. See how our small model stacks up against the big frontier models

In [ ]:
!pip install -q --upgrade bitsandbytes trl openai
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
import re
import statistics
from google.colab import userdata
from huggingface_hub import login
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset
from peft import PeftModel
from util import Tester

# --- Config ---
BASE_MODEL = "meta-llama/Llama-3.2-3B"
HF_USER = "mcaleb"
DATASET_NAME = "ed-donner/items_prompts_lite"
RUN_NAME = "2026-03-05_04.52.30-lite"
HUB_MODEL_NAME = f"{HF_USER}/price-{RUN_NAME}"

SIZE = 200

def run_eval(predictor, data, title=None, size=SIZE):
    """Run the course's Tester (shows charts) and return the average error."""
    t = Tester(predictor, data, title=title, size=size)
    t.run()
    return sum(t.errors) / t.size

In [ ]:
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

dataset = load_dataset(DATASET_NAME)
test = dataset["test"]

## Loading our fine-tuned model

4-bit quantized Llama 3.2-3B with our QLoRA adapter on top.

In [ ]:
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

## Approach 1: Greedy — just pick the best guess

The simplest way to get a prediction. The model looks at the product description
and generates the most likely next tokens one by one. Whatever comes out first is our answer.

In [ ]:
def greedy_predict(item):
    """Feed the prompt to the model, let it generate a price."""
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output_ids[0, prompt_len:])

results = {}
set_seed(42)
results["QLoRA Greedy"] = run_eval(greedy_predict, test, "QLoRA Greedy", SIZE)

## Approach 2: Ask multiple times and take the middle answer

Instead of asking the model once, we ask it 3 times with a bit of randomness
(temperature=0.2). Then we pick the middle answer. This helps smooth out
cases where one unlucky guess throws things off.

In [ ]:
def ensemble_predict(item, n_samples=3, temperature=0.2):
    """Generate several predictions with some randomness, return the median."""
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    prices = []
    for _ in range(n_samples):
        with torch.no_grad():
            output_ids = fine_tuned_model.generate(
                **inputs, max_new_tokens=8,
                do_sample=True, temperature=temperature,
            )
        text = tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:])
        # pull out the number from the generated text
        match = re.search(r"\d+\.?\d*", text)
        if match:
            prices.append(float(match.group()))
    return str(statistics.median(prices)) if prices else "0"

set_seed(42)
results["QLoRA Ensemble"] = run_eval(ensemble_predict, test, "QLoRA Ensemble", SIZE)

## Approach 3: Blend the model's top guesses

Instead of generating text, we look directly at the model's probability distribution
for the first price token. We take the top 3 most likely tokens, check which ones
are valid numbers, and compute a weighted average based on how confident the model
is in each one. This is like asking "what are your best guesses and how sure are you?"

In [ ]:
# Need float32 for the logit math (changes the model in-place)
fine_tuned_model = fine_tuned_model.float()

def topk_predict(item, k=3):
    """Look at the model's top-K token probabilities and blend them."""
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = fine_tuned_model(**inputs)
        logits = outputs.logits[:, -1, :].to("cpu")

    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(k)

    # keep only tokens that decode to valid prices
    prices, weights = [], []
    for i in range(k):
        token = tokenizer.decode(top_ids[0][i]).strip()
        try:
            price = float(token)
            if price > 0:
                prices.append(price)
                weights.append(top_probs[0][i])
        except ValueError:
            continue

    if not prices:
        return "0"
    total = sum(weights)
    return str(sum(p * w / total for p, w in zip(prices, weights)).item())

set_seed(42)
results["QLoRA Top-K"] = run_eval(topk_predict, test, "QLoRA Top-K", SIZE)

## How do frontier models compare?

Let's test 4 frontier models on the same 200 items:
- **GPT-4.1-nano** — OpenAI's smallest model, zero-shot (no training)
- **GPT-4.1-nano fine-tuned** — same model but fine-tuned on 2K examples (from week 6)
- **Claude Haiku** — Anthropic's fast model, zero-shot
- **Gemini Flash** — Google's fast model, zero-shot

In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY"),
)

def extract_summary(item):
    p = item["prompt"]
    return p.split("What does this cost to the nearest dollar?\n\n")[1].split("\n\nPrice is $")[0]

def make_frontier_predictor(client, model):
    def predict(item):
        summary = extract_summary(item)
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content":
                f"Estimate the price of this product. Respond with just the dollar amount, no explanation.\n\n{summary}"}],
            max_tokens=10,
            temperature=0,
        )
        return r.choices[0].message.content
    predict.__name__ = model.split("/")[-1].split(":")[0]
    return predict

frontier_models = {
    "GPT-4.1-nano": (openai_client, "gpt-4.1-nano"),
    "GPT-4.1-nano FT": (openai_client, "ft:gpt-4.1-nano-2025-04-14:personal:pricer-2000:DFdADQPs"),
    "Claude Haiku": (openrouter_client, "anthropic/claude-3.5-haiku"),
    "Gemini Flash": (openrouter_client, "google/gemini-2.5-flash-lite"),
}

In [ ]:
for name, (client, model) in frontier_models.items():
    predictor = make_frontier_predictor(client, model)
    set_seed(42)
    results[name] = run_eval(predictor, test, name, SIZE)
    print(f"{name}: ${results[name]:.2f}")

## Final comparison

In [ ]:
import plotly.graph_objects as go

labels = list(results.keys())
values = list(results.values())
colors = ["red" if "QLoRA" in k else "skyblue" if "FT" in k else "slateblue" for k in labels]

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
fig.update_layout(
    title="Price Prediction: Mean Absolute Error by Model",
    yaxis=dict(title="MAE ($)", range=[0, max(values) * 1.1]),
    xaxis=dict(tickangle=-30),
    width=900,
    height=500,
)
fig.show()

print(f"\n{'Model':<25} {'MAE':>10}")
print("-" * 37)
for name, mae in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:<25} ${mae:>8.2f}")

## What we learned

- **Greedy decoding** ($65) is a decent starting point but leaves room on the table.
- **Ensemble** ($57) — asking 3 times with low temperature and taking the median
  gave the single biggest improvement. Simple idea, big payoff.
- **Top-K blending** ($57) — looking at the probability distribution directly
  works about as well as the ensemble, through a completely different mechanism.
- Both tricks brought our small QLoRA model into the same ballpark as frontier
  models like Claude Haiku ($58) and Gemini Flash ($56) — without any API costs.
- **Fine-tuning wins**: GPT-4.1-nano FT ($54) still leads, showing that even
  a tiny fine-tuned frontier model is hard to beat.